# King County: Refactored Data Preparation and Feature Engineering

This is a refactored copy of `King-County.ipynb`. It keeps only the logic needed to turn raw sales data into model-ready features:

- **Removed:** exploration (`describe()`, plots, maps), one-off inspections, and the modeling section (now in `src/king_county/modeling.py`).
- **Refactored:** every cleaning and feature step becomes a small function with one job, clear names, no magic numbers, and no side effects (`df.copy()` first, DataFrame in, DataFrame out).
- **Moved:** the final functions live in `src/king_county/`, and the last section of this notebook uses them through `HousingPipeline`.

```mermaid
flowchart LR
    A[Raw CSV] --> B[Data preparation] --> C[Feature engineering] --> D[Model-ready DataFrame]
```

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# numpy for vectorized distance math, pandas for the tabular data.
import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal

RAW_DATA_PATH = "data/King_County_House_prices_dataset.csv"

raw_houses = pd.read_csv(RAW_DATA_PATH)
raw_houses.shape

## 1. Data Preparation

Each notebook cleaning step is rewritten as a reusable rule.

> **Checkpoint:** each function owns exactly one transformation.

### 1.1 Remove the bedroom outlier

The original notebook dropped one row by its index (`drop(15856)`). That breaks as soon as the data changes. Instead we encode the reason it is wrong: a house cannot have less than 100 sqft of living area per bedroom. In the full dataset this removes only the 33-bedroom house (49 sqft per bedroom); the next-smallest value is about 273.

In [ ]:
MIN_SQFT_PER_BEDROOM = 100


def remove_bedroom_outlier(df: pd.DataFrame) -> pd.DataFrame:
    """Drop houses whose living area is too small for their number of bedrooms."""
    df = df.copy()
    sqft_per_bedroom = df["sqft_living"] / df["bedrooms"]
    return df.loc[sqft_per_bedroom >= MIN_SQFT_PER_BEDROOM].copy()

### 1.2 Rebuild `sqft_basement`

The raw column contains `?`. Basement area is always living area minus above-ground area, so we recompute it instead of patching the text values.

In [ ]:
def sqft_basement(df: pd.DataFrame) -> pd.DataFrame:
    """Rebuild basement size from living and above-ground area."""
    df = df.copy()
    df["sqft_basement"] = df["sqft_living"] - df["sqft_above"]
    return df

### 1.3 Fill missing `view` and `waterfront`

Missing values are treated as "no view" and "not on the water", like in the original notebook.

In [ ]:
def fill_missings_view_wf(df: pd.DataFrame) -> pd.DataFrame:
    """Fill missing view and waterfront values with 0."""
    df = df.copy()
    df["view"] = df["view"].fillna(0)
    df["waterfront"] = df["waterfront"].fillna(0)
    return df

### 1.4 Create `last_known_change`

The original used a row-by-row loop. The vectorized version gives the same result: the renovation year if the house was renovated, otherwise the build year.

In [ ]:
def calculate_last_change(df: pd.DataFrame) -> pd.DataFrame:
    """Combine yr_renovated and yr_built into last_known_change."""
    df = df.copy()
    renovated = df["yr_renovated"].fillna(0)
    df["last_known_change"] = renovated.where(renovated != 0, df["yr_built"]).astype(int)
    return df.drop(columns=["yr_renovated", "yr_built"])

In [ ]:
def prepare_data(df: pd.DataFrame) -> pd.DataFrame:
    """Run all cleaning steps in the same order as the original notebook."""
    df = remove_bedroom_outlier(df)
    df = sqft_basement(df)
    df = fill_missings_view_wf(df)
    df = calculate_last_change(df)
    return df


cleaned_houses = prepare_data(raw_houses)
print(cleaned_houses.shape)
print("Missing view/waterfront:", cleaned_houses[["view", "waterfront"]].isna().sum().sum())

## 2. Feature Engineering

The magic numbers from the original cells become named constants.

In [ ]:
WEALTH_CENTER_LAT = 47.62774      # Bill Gates' house in Medina
WEALTH_CENTER_LONG = -122.24194
LONG_CORRECTION_LAT = 47.6219     # latitude used for the cos(lat) correction
EARTH_RADIUS_KM = 6378
KM_PER_DEGREE = 2 * np.pi * EARTH_RADIUS_KM / 360


def distance_km(long, lat, ref_long, ref_lat):
    """Approximate distance in km between points and a reference location."""
    delta_long_corr = (long - ref_long) * np.cos(np.radians(ref_lat))
    delta_lat = lat - ref_lat
    return np.sqrt(delta_long_corr**2 + delta_lat**2) * KM_PER_DEGREE

### 2.1 Price per square foot

In [ ]:
def add_sqft_price(df: pd.DataFrame) -> pd.DataFrame:
    """Add price per square foot of living area plus lot."""
    df = df.copy()
    df["sqft_price"] = (df["price"] / (df["sqft_living"] + df["sqft_lot"])).round(2)
    return df

### 2.2 Distance to the center of wealth

In [ ]:
def add_center_distance(df: pd.DataFrame) -> pd.DataFrame:
    """Add the distance in km to the wealth center in Medina."""
    df = df.copy()
    df["delta_lat"] = np.absolute(WEALTH_CENTER_LAT - df["lat"])
    df["delta_long"] = np.absolute(WEALTH_CENTER_LONG - df["long"])
    df["center_distance"] = (
        np.sqrt(
            (df["delta_long"] * np.cos(np.radians(LONG_CORRECTION_LAT))) ** 2
            + df["delta_lat"] ** 2
        )
        * KM_PER_DEGREE
    )
    return df

### 2.3 Distance to the nearest waterfront house

The original used a loop inside a loop (21,000 × 146 iterations). Numpy broadcasting computes all distances at once. The waterfront locations are an explicit input, so the same function also works for a single new house (as in the API).

In [ ]:
def get_waterfront_locations(df: pd.DataFrame) -> pd.DataFrame:
    """Return the lat/long of all waterfront houses, used as water reference points."""
    return df.loc[df["waterfront"] == 1, ["lat", "long"]].reset_index(drop=True)


def add_water_distance(df: pd.DataFrame, waterfront: pd.DataFrame) -> pd.DataFrame:
    """Add the distance in km to the nearest house in the waterfront reference set."""
    df = df.copy()
    distances = distance_km(
        df["long"].to_numpy()[:, None],
        df["lat"].to_numpy()[:, None],
        waterfront["long"].to_numpy()[None, :],
        waterfront["lat"].to_numpy()[None, :],
    )
    df["water_distance"] = distances.min(axis=1)
    return df

In [ ]:
def engineer_features(df: pd.DataFrame, waterfront: pd.DataFrame | None = None) -> pd.DataFrame:
    """Run all feature engineering steps in the same order as the original notebook."""
    if waterfront is None:
        waterfront = get_waterfront_locations(df)
    df = add_sqft_price(df)
    df = add_center_distance(df)
    df = add_water_distance(df, waterfront)
    return df


houses = engineer_features(cleaned_houses)
houses.shape

## 3. From Notebook to Python Scripts

The functions above were moved unchanged into:

| File | Contents |
|---|---|
| `src/king_county/data_preparation.py` | `load_data`, the 4 cleaning functions, `prepare_data` |
| `src/king_county/feature_engineering.py` | constants, distance and feature functions, `engineer_features` |
| `src/king_county/pipeline.py` | `HousingPipeline`: read → process → write |

The cell below checks that the packaged pipeline produces exactly the same DataFrame as the notebook functions.

In [ ]:
from src.king_county.pipeline import PROCESSED_DATA_PATH, HousingPipeline

pipeline_houses = HousingPipeline(RAW_DATA_PATH, PROCESSED_DATA_PATH).run()

assert_frame_equal(houses, pipeline_houses)
print("Notebook functions and packaged pipeline give identical results:", pipeline_houses.shape)

## 4. Modeling

Modeling was also moved to `src/king_county/modeling.py`. It is run from the terminal:

```bash
uv run python -m src.king_county.modeling
```

This prepares the data with `HousingPipeline`, tunes the ElasticNet pipeline with `GridSearchCV`, evaluates it on the test set (R² ≈ 0.84, MAE ≈ $95k), and saves `model/model.bin` plus `model/waterfront_locations.csv` for the API.